
# Priority Product Test Group Creation

## Objective

Construct five **balanced** test groups of OD markets for the Priority product pricing experiment — one **Control** arm and four **treatment** arms (T1–T4) — so that the groups are as comparable as possible *before* any treatment is applied.

Because the Priority product is assigned at the **OD-market level** (not at the flight, offer, or passenger level), simple randomization can easily produce arms that differ in market composition. If one arm happens to contain more high-volume, weekend-heavy, or long-haul markets than another, those differences would confound the treatment effect. This notebook therefore uses **stratified assignment** on market characteristics to minimize pre-treatment imbalance, so that any post-launch performance difference can be credibly attributed to the price treatment itself rather than to which markets landed in which arm.

## Treatment Arms

| Group | Price multiplier | Description |
|-------|-----------------:|-------------|
| Control | 1.00 | No change |
| T1 | 0.85 | −15% |
| T2 | 0.90 | −10% |
| T3 | 1.10 | +10% |
| T4 | 1.15 | +15% |

## Experimental Unit

**Unit of assignment:** OD Market (e.g., `DFW-MIA`, `CLT-MCO`, `ORD-DCA`). Each OD market is assigned to exactly one test group.

## Approach

1. **Feature engineering** — characterize every OD market on the dimensions that could bias results: Priority Group, flight-duration category, day-of-week (DOW) demand profile, and volume (PNRs / passengers).
2. **Stratified assignment** — within each `priority_group × flight_duration` stratum, shuffle and round-robin markets across the five arms so each arm receives a structurally similar mix.
3. **Balance validation** — quantify residual imbalance across all criteria before trusting the split.
4. **Pilot selection** — split each arm into folds and search fold combinations to carve out a smaller, still-balanced pilot subset for a low-risk initial launch.
5. **Deployment plan** — map the selected markets to travel dates and emit rows in the exact format the pricing tool ingests (treatment multiplier per OD).

## Outputs

- `priority_test_group_assignment` — full OD-to-arm assignment.
- `priority_pilot_plan` — deployment-ready pilot plan (OD, travel dates, treatment multiplier).

---

## Desired Balance Criteria

The five groups should be balanced with respect to the following characteristics:

### 1. Number of OD Markets

Each group should contain approximately the same number of OD markets.

| Group | OD Count |
|---------|---------:|
| Control | ~20% |
| T1 | ~20% |
| T2 | ~20% |
| T3 | ~20% |
| T4 | ~20% |

The goal is to avoid test groups that are substantially larger or smaller than others.

---

### 2. Priority Group Mix

Each treatment arm should contain a similar distribution of Priority Groups.

| Group | PG1 | PG2 | PG3 | PG4 | PG5 | PG6 |
|---------|-----:|-----:|-----:|-----:|-----:|-----:|
| Control | x% | x% | x% | x% | x% | x% |
| T1 | x% | x% | x% | x% | x% | x% |
| T2 | x% | x% | x% | x% | x% | x% |
| T3 | x% | x% | x% | x% | x% | x% |
| T4 | x% | x% | x% | x% | x% | x% |

No treatment group should be disproportionately concentrated in a specific Priority Group.

---

### 3. Flight Duration Mix

Each treatment arm should contain a similar distribution of flight-duration categories.

| Group | Ultra Short | Short | Medium | Long |
|---------|------------:|-------:|-------:|------:|
| Control | x% | x% | x% | x% |
| T1 | x% | x% | x% | x% |
| T2 | x% | x% | x% | x% |
| T3 | x% | x% | x% | x% |
| T4 | x% | x% | x% | x% |

The objective is to avoid systematic differences in trip length across treatment groups.

---

### 4. DOW Profile Mix

The goal is not necessarily to balance individual departures by weekday. Instead, the goal is to balance the mix of market-level DOW profiles (e.g., weekday-heavy, weekend-heavy, or balanced markets).

| Group | Avg Mon | Avg Tue | Avg Wed | Avg Thu | Avg Fri | Avg Sat | Avg Sun |
|---------|---------:|---------:|---------:|---------:|---------:|---------:|---------:|
| Control | x% | x% | x% | x% | x% | x% | x% |
| T1 | x% | x% | x% | x% | x% | x% | x% |
| T2 | x% | x% | x% | x% | x% | x% | x% |
| T3 | x% | x% | x% | x% | x% | x% | x% |
| T4 | x% | x% | x% | x% | x% | x% | x% |

Alternatively, DOW profiles may be collapsed into broader weekday-versus-weekend measures if that provides a more interpretable balancing target.

---


In [1]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import itertools

RANDOM_SEED = 42

GROUPS = [
    "Control",
    "T1_minus15",
    "T2_minus10",
    "T3_plus10",
    "T4_plus15"
]

DOW_COLS = [
    "pct_sun",
    "pct_mon",
    "pct_tue",
    "pct_wed",
    "pct_thu",
    "pct_fri",
    "pct_sat"
]

N_FOLDS = 10

TREATMENT_VALUE_MAP = {
    "Control": 1.00,
    "T1_minus15": 0.85,
    "T2_minus10": 0.90,
    "T3_plus10": 1.10,
    "T4_plus15": 1.15
}

# July 14 2026: Need to be adjusted. Ask Sam about how long exploration runs in general
START_DATE = "2026-08-01"
END_DATE = "2026-08-14"

In [2]:
priority_airport = pd.read_excel("Priority_Group_Airport.xlsx")

priority_airport_spark = spark.createDataFrame(priority_airport)
priority_airport_spark.createOrReplaceTempView("priority_airport")

finalTransactionOfferSale = spark.table("rm_workspace.finalTransactionOfferSale_B")
finalTransactionOfferSale.createOrReplaceTempView("finalTransactionOfferSale")

print(f"finalTransactionOfferSale_B: {finalTransactionOfferSale.count():,} rows")

finalTransactionOfferSale_B: 52,582,746 rows


In [3]:
itinerary = (
    spark.table("rm_workspace.tmp_pnr_spine")
    .filter(F.length(F.col("fare_basis_cd")) <= 8)
    .filter(F.col("pax_count") > 0)
)

itinerary.createOrReplaceTempView("itinerary")

In [4]:
priority_airport_sdf = spark.table("priority_airport")
finalTransactionOfferSale = spark.table("finalTransactionOfferSale")

offer_sale = (
    finalTransactionOfferSale.alias("f")
    .join(
        priority_airport_sdf.alias("p"),
        F.col("f.od_origin") == F.col("p.Airport"),
        "left"
    )
    .withColumn(
        "DOW",
        F.dayofweek("OD_dep_dt")
    )
    .withColumn(
        "Priority_Group",
        F.coalesce(F.col("p.Group"), F.lit(6))
    )
)

offer_sale.createOrReplaceTempView("offer_sale")

In [5]:
dow_counts = (
    itinerary
    .filter(F.col("region").like("%US48%"))
    .groupBy(
        F.col("od_origin_airprt_iata_cd").alias("od_origin"),
        F.col("od_destntn_airprt_iata_cd").alias("od_destination"),
        F.dayofweek("od_local_dep_dt").alias("dow")
    )
    .agg(
        F.count("*").alias("dep_cnt")
    )
)

totals = (
    dow_counts
    .groupBy("od_origin", "od_destination")
    .agg(
        F.sum("dep_cnt").alias("total_dep")
    )
)

od_dow = (
    dow_counts.alias("d")
    .join(
        totals.alias("t"),
        ["od_origin", "od_destination"]
    )
    .groupBy("od_origin", "od_destination")
    .agg(
        F.round(
            100 * F.sum(F.when(F.col("dow") == 1, F.col("dep_cnt")).otherwise(0))
            / F.max("total_dep"),
            2
        ).alias("pct_sun"),

        F.round(
            100 * F.sum(F.when(F.col("dow") == 2, F.col("dep_cnt")).otherwise(0))
            / F.max("total_dep"),
            2
        ).alias("pct_mon"),

        F.round(
            100 * F.sum(F.when(F.col("dow") == 3, F.col("dep_cnt")).otherwise(0))
            / F.max("total_dep"),
            2
        ).alias("pct_tue"),

        F.round(
            100 * F.sum(F.when(F.col("dow") == 4, F.col("dep_cnt")).otherwise(0))
            / F.max("total_dep"),
            2
        ).alias("pct_wed"),

        F.round(
            100 * F.sum(F.when(F.col("dow") == 5, F.col("dep_cnt")).otherwise(0))
            / F.max("total_dep"),
            2
        ).alias("pct_thu"),

        F.round(
            100 * F.sum(F.when(F.col("dow") == 6, F.col("dep_cnt")).otherwise(0))
            / F.max("total_dep"),
            2
        ).alias("pct_fri"),

        F.round(
            100 * F.sum(F.when(F.col("dow") == 7, F.col("dep_cnt")).otherwise(0))
            / F.max("total_dep"),
            2
        ).alias("pct_sat")
    )
)

od_dow.createOrReplaceTempView("od_dow")

In [6]:
pnr_od = (
    itinerary
    .filter(F.col("region").like("%US48%"))
    .groupBy(
        F.col("od_origin_airprt_iata_cd").alias("od_origin"),
        F.col("od_destntn_airprt_iata_cd").alias("od_destination"),
        "pnr_loctr_id"
    )
    .agg(
        F.max("pax_count").alias("pax_count")
    )
)

od_volume = (
    pnr_od
    .groupBy(
        "od_origin",
        "od_destination"
    )
    .agg(
        F.countDistinct("pnr_loctr_id").alias("pnr_cnt"),
        F.sum("pax_count").alias("pax_cnt")
    )
)

od_volume.createOrReplaceTempView("od_volume")

In [7]:
od_market_features = (
    offer_sale.alias("o")
    .join(
        od_dow.alias("d"),
        ["od_origin", "od_destination"],
        "left"
    )
    .join(
        od_volume.alias("v"),
        ["od_origin", "od_destination"],
        "left"
    )
    .filter(
        F.col("region_group") == "Domestic"
    )
    .groupBy(
        "od_origin",
        "od_destination",
        "pct_sun",
        "pct_mon",
        "pct_tue",
        "pct_wed",
        "pct_thu",
        "pct_fri",
        "pct_sat",
        "pnr_cnt",
        "pax_cnt"
    )
    .agg(
        F.max("Priority_Group").alias("priority_group"),
        F.max("FlightDuration").alias("flight_duration")
    )
    .withColumn(
        "pnr_cnt",
        F.coalesce(F.col("pnr_cnt"), F.lit(0))
    )
    .withColumn(
        "pax_cnt",
        F.coalesce(F.col("pax_cnt"), F.lit(0))
    )
)

od_market_features.createOrReplaceTempView("od_market_features")

In [8]:
od_features = spark.sql("""
SELECT
    od_origin,
    od_destination,
    priority_group,
    flight_duration,
    pct_sun,
    pct_mon,
    pct_tue,
    pct_wed,
    pct_thu,
    pct_fri,
    pct_sat,
    pnr_cnt,
    pax_cnt
FROM od_market_features
""").toPandas()

# Force numeric types after Spark -> pandas conversion
for c in DOW_COLS:
    od_features[c] = pd.to_numeric(od_features[c], errors="coerce")

for c in ["pnr_cnt", "pax_cnt"]:
    od_features[c] = pd.to_numeric(od_features[c], errors="coerce").fillna(0)

# Optional: if any OD has missing DOW because od_dow did not join
od_features[DOW_COLS] = od_features[DOW_COLS].fillna(0)

display(od_features[DOW_COLS].describe())


od_features["stratum"] = (
    od_features["priority_group"].astype(str)
    + "_"
    + od_features["flight_duration"].astype(str)
)

od_features = (
    od_features
    .groupby("stratum", group_keys=False)
    .apply(lambda x: x.sample(frac=1, random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

od_features["group_idx"] = (
    od_features
    .groupby("stratum")
    .cumcount()
    % len(GROUPS)
)

od_features["test_group"] = (
    od_features["group_idx"]
    .map(dict(enumerate(GROUPS)))
)

,pct_sun,pct_mon,pct_tue,pct_wed,pct_thu,pct_fri,pct_sat
count,50644.000000,50644.000000,50644.000000,50644.000000,50644.000000,50644.000000,50644.000000
mean,14.160848,14.677802,12.255283,12.854194,14.301234,13.990542,11.873752
std,9.603221,9.850276,9.085258,9.149757,9.397284,9.410017,9.683068
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,10.340000,10.710000,8.850000,9.520000,10.590000,10.340000,7.247500
50%,14.410000,14.790000,11.970000,12.850000,14.700000,14.290000,11.110000
75%,17.650000,18.232500,14.810000,15.550000,17.850000,17.300000,14.860000
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


C:\Users\939510\AppData\Local\Temp\1\ipykernel_10720\966882314.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=1, random_state=RANDOM_SEED))


In [9]:
def validate_assignment(df, title="BALANCE SUMMARY"):
    print("=" * 70)
    print(title)
    print("=" * 70)

    count_balance = df.groupby("test_group").size()
    print("\nOD count by group:")
    print(count_balance)
    print("\nOD count max-min:")
    print(count_balance.max() - count_balance.min())

    pg_balance = pd.crosstab(
        df["test_group"],
        df["priority_group"],
        normalize="index"
    ) * 100

    print("\nPriority Group mix:")
    display(pg_balance.round(2))

    print("\nPriority Group max-min percentage point imbalance:")
    print((pg_balance.max() - pg_balance.min()).round(3))

    fd_balance = pd.crosstab(
        df["test_group"],
        df["flight_duration"],
        normalize="index"
    ) * 100

    print("\nFlight Duration mix:")
    display(fd_balance.round(2))

    print("\nFlight Duration max-min percentage point imbalance:")
    print((fd_balance.max() - fd_balance.min()).round(3))

    dow_balance = df.groupby("test_group")[DOW_COLS].mean()

    print("\nDOW profile:")
    display(dow_balance.round(2))

    print("\nDOW max-min percentage point imbalance:")
    print((dow_balance.max() - dow_balance.min()).round(3))

    volume_summary = (
        df
        .groupby("test_group")
        .agg(
            n_ods=("od_origin", "size"),
            total_pnrs=("pnr_cnt", "sum"),
            avg_pnrs_per_od=("pnr_cnt", "mean"),
            median_pnrs_per_od=("pnr_cnt", "median"),
            total_pax=("pax_cnt", "sum"),
            avg_pax_per_od=("pax_cnt", "mean"),
            median_pax_per_od=("pax_cnt", "median")
        )
        .round(2)
    )

    print("\nVolume summary:")
    display(volume_summary)

    pnr_spread = volume_summary["total_pnrs"].max() - volume_summary["total_pnrs"].min()
    pnr_spread_pct = 100 * pnr_spread / volume_summary["total_pnrs"].mean()

    pax_spread = volume_summary["total_pax"].max() - volume_summary["total_pax"].min()
    pax_spread_pct = 100 * pax_spread / volume_summary["total_pax"].mean()

    print("\nPNR spread:")
    print(f"{pnr_spread:,.0f} PNRs ({pnr_spread_pct:.2f}% of mean)")

    print("\nPAX spread:")
    print(f"{pax_spread:,.0f} PAX ({pax_spread_pct:.2f}% of mean)")

    return {
        "count_balance": count_balance,
        "pg_balance": pg_balance,
        "fd_balance": fd_balance,
        "dow_balance": dow_balance,
        "volume_summary": volume_summary
    }


full_validation = validate_assignment(
    od_features,
    title="FULL UNIVERSE BALANCE SUMMARY"
)

FULL UNIVERSE BALANCE SUMMARY

OD count by group:
test_group
Control       10142
T1_minus15    10134
T2_minus10    10127
T3_plus10     10124
T4_plus15     10117
dtype: int64

OD count max-min:
25

Priority Group mix:


priority_group,1,2,3,4,5,6
test_group,,,,,,
Control,3.75,16.34,34.16,28.40,7.30,10.06
T1_minus15,3.74,16.33,34.17,28.42,7.29,10.05
T2_minus10,3.71,16.34,34.20,28.42,7.30,10.03
T3_plus10,3.71,16.35,34.20,28.43,7.29,10.03
T4_plus15,3.72,16.35,34.21,28.43,7.26,10.03



Priority Group max-min percentage point imbalance:
priority_group
1    0.034
2    0.018
3    0.045
4    0.031
5    0.032
6    0.032
dtype: float64

Flight Duration mix:


flight_duration,Medium,Short,True_Long,Ultra_Short
test_group,,,,
Control,10.65,38.84,0.14,50.37
T1_minus15,10.66,38.84,0.12,50.39
T2_minus10,10.65,38.85,0.11,50.40
T3_plus10,10.65,38.84,0.11,50.41
T4_plus15,10.63,38.87,0.08,50.42



Flight Duration max-min percentage point imbalance:
flight_duration
Medium         0.026
Short          0.030
True_Long      0.059
Ultra_Short    0.050
dtype: float64

DOW profile:


,pct_sun,pct_mon,pct_tue,pct_wed,pct_thu,pct_fri,pct_sat
test_group,,,,,,,
Control,14.18,14.69,12.27,12.77,14.33,13.92,11.91
T1_minus15,14.28,14.65,12.15,12.91,14.28,14.16,11.82
T2_minus10,14.09,14.64,12.24,12.81,14.29,13.79,11.85
T3_plus10,14.21,14.65,12.34,12.87,14.33,14.04,11.97
T4_plus15,14.04,14.75,12.28,12.90,14.28,14.04,11.81



DOW max-min percentage point imbalance:
pct_sun    0.239
pct_mon    0.105
pct_tue    0.184
pct_wed    0.143
pct_thu    0.052
pct_fri    0.365
pct_sat    0.157
dtype: float64

Volume summary:


,n_ods,total_pnrs,avg_pnrs_per_od,median_pnrs_per_od,total_pax,avg_pax_per_od,median_pax_per_od
test_group,,,,,,,
Control,10142,12096309,1192.69,121.0,12096470,1192.71,121.0
T1_minus15,10134,11278340,1112.92,122.0,11278447,1112.93,122.0
T2_minus10,10127,12016471,1186.58,113.0,12016596,1186.59,113.0
T3_plus10,10124,12048760,1190.12,117.0,12048902,1190.13,117.0
T4_plus15,10117,11911375,1177.36,119.0,11911515,1177.38,119.0



PNR spread:
817,969 PNRs (6.89% of mean)

PAX spread:
818,023 PAX (6.89% of mean)


In [10]:
final_assignment = od_features[
    [
        "od_origin",
        "od_destination",
        "priority_group",
        "flight_duration",
        "test_group",
        "pnr_cnt",
        "pax_cnt"
    ]
].copy()

final_assignment_spark = spark.createDataFrame(final_assignment)

final_assignment_spark.createOrReplaceTempView(
    "priority_test_group_assignment"
)


# final_assignment_spark.write.mode("overwrite").saveAsTable(
#     "rm_workspace.priority_test_group_assignment"
# )

In [11]:
od_features["pnr_bucket"] = pd.qcut(
    od_features["pnr_cnt"],
    q=10,
    labels=False,
    duplicates="drop"
)

fold_strata_cols = [
    "test_group",
    "priority_group",
    "flight_duration",
    "pnr_bucket"
]

df = od_features.copy()

rng = np.random.default_rng(RANDOM_SEED)
df["_rand"] = rng.random(len(df))

df = (
    df
    .sort_values(
        fold_strata_cols + ["pnr_cnt", "_rand"],
        ascending=[True, True, True, True, False, True]
    )
    .reset_index(drop=True)
)

df["_rank_in_stratum"] = (
    df
    .groupby(fold_strata_cols, dropna=False)
    .cumcount()
)

df["pilot_fold"] = df["_rank_in_stratum"] % N_FOLDS

In [12]:
fold_summary = (
    df
    .groupby(["test_group", "pilot_fold"])
    .agg(
        n_ods=("od_origin", "size"),
        total_pnrs=("pnr_cnt", "sum"),
        total_pax=("pax_cnt", "sum"),
        pct_sun=("pct_sun", "mean"),
        pct_mon=("pct_mon", "mean"),
        pct_tue=("pct_tue", "mean"),
        pct_wed=("pct_wed", "mean"),
        pct_thu=("pct_thu", "mean"),
        pct_fri=("pct_fri", "mean"),
        pct_sat=("pct_sat", "mean")
    )
    .reset_index()
)


best_combo = None
best_score = np.inf
best_combo_summary = None

for combo in itertools.product(range(N_FOLDS), repeat=len(GROUPS)):

    selected_rows = []

    for group, fold in zip(GROUPS, combo):
        row = fold_summary[
            (fold_summary["test_group"] == group)
            & (fold_summary["pilot_fold"] == fold)
        ]
        selected_rows.append(row)

    combo_summary = pd.concat(selected_rows, ignore_index=True)

    pnr_spread_pct = (
        100
        * (
            combo_summary["total_pnrs"].max()
            - combo_summary["total_pnrs"].min()
        )
        / combo_summary["total_pnrs"].mean()
    )

    pax_spread_pct = (
        100
        * (
            combo_summary["total_pax"].max()
            - combo_summary["total_pax"].min()
        )
        / combo_summary["total_pax"].mean()
    )

    count_spread_pct = (
        100
        * (
            combo_summary["n_ods"].max()
            - combo_summary["n_ods"].min()
        )
        / combo_summary["n_ods"].mean()
    )

    dow_spread_avg_pp = (
        combo_summary[DOW_COLS].max()
        - combo_summary[DOW_COLS].min()
    ).mean()

    score = (
        5.0 * pnr_spread_pct
        + 5.0 * pax_spread_pct
        + 1.0 * count_spread_pct
        + 2.0 * dow_spread_avg_pp
    )

    if score < best_score:
        best_score = score
        best_combo = combo
        best_combo_summary = combo_summary.copy()

print("=" * 70)
print("BEST PILOT FOLD COMBINATION")
print("=" * 70)
print("Best combo:")
print(dict(zip(GROUPS, best_combo)))
print(f"Best score: {best_score:.4f}")

display(best_combo_summary)

BEST PILOT FOLD COMBINATION
Best combo:
{'Control': 4, 'T1_minus15': 3, 'T2_minus10': 4, 'T3_plus10': 4, 'T4_plus15': 4}
Best score: 41.0716


,test_group,pilot_fold,n_ods,total_pnrs,total_pax,pct_sun,pct_mon,pct_tue,pct_wed,pct_thu,pct_fri,pct_sat
0,Control,4,1022,1166146,1166160,13.707094,14.953043,12.183679,13.258542,14.344031,13.881703,11.996595
1,T1_minus15,3,1039,1163922,1163930,14.463465,14.708191,11.882820,13.065910,14.283109,14.008219,11.909211
2,T2_minus10,4,1018,1125168,1125178,13.550491,14.738016,12.749941,12.987279,14.631690,13.698910,11.356572
3,T3_plus10,4,1021,1166242,1166255,14.527512,14.327463,12.556856,13.032106,14.180823,13.695416,11.901352
4,T4_plus15,4,1015,1123291,1123305,14.027015,14.861222,12.430729,12.839222,14.005714,14.242631,11.878887


In [13]:
selected_parts = []

for group, fold in zip(GROUPS, best_combo):
    selected_parts.append(
        df[
            (df["test_group"] == group)
            & (df["pilot_fold"] == fold)
        ]
    )

pilot_od_features = (
    pd.concat(selected_parts, ignore_index=True)
    .reset_index(drop=True)
)

In [14]:
pilot_validation = validate_assignment(
    pilot_od_features,
    title="PILOT SUBSET BALANCE SUMMARY"
)

PILOT SUBSET BALANCE SUMMARY

OD count by group:
test_group
Control       1022
T1_minus15    1039
T2_minus10    1018
T3_plus10     1021
T4_plus15     1015
dtype: int64

OD count max-min:
24

Priority Group mix:


priority_group,1,2,3,4,5,6
test_group,,,,,,
Control,3.91,16.54,34.05,28.38,7.24,9.88
T1_minus15,3.95,16.27,33.78,28.20,7.60,10.20
T2_minus10,3.73,16.40,34.18,28.29,7.47,9.92
T3_plus10,3.82,16.26,33.69,28.31,7.74,10.19
T4_plus15,3.74,16.55,34.19,28.67,7.00,9.85



Priority Group max-min percentage point imbalance:
priority_group
1    0.213
2    0.293
3    0.495
4    0.470
5    0.742
6    0.350
dtype: float64

Flight Duration mix:


flight_duration,Medium,Short,True_Long,Ultra_Short
test_group,,,,
Control,10.96,38.85,0.1,50.10
T1_minus15,11.16,38.40,0.1,50.34
T2_minus10,10.71,39.10,0.1,50.10
T3_plus10,10.68,38.69,0.1,50.54
T4_plus15,10.64,38.82,0.1,50.44



Flight Duration max-min percentage point imbalance:
flight_duration
Medium         0.524
Short          0.694
True_Long      0.002
Ultra_Short    0.441
dtype: float64

DOW profile:


,pct_sun,pct_mon,pct_tue,pct_wed,pct_thu,pct_fri,pct_sat
test_group,,,,,,,
Control,13.71,14.95,12.18,13.26,14.34,13.88,12.00
T1_minus15,14.46,14.71,11.88,13.07,14.28,14.01,11.91
T2_minus10,13.55,14.74,12.75,12.99,14.63,13.70,11.36
T3_plus10,14.53,14.33,12.56,13.03,14.18,13.70,11.90
T4_plus15,14.03,14.86,12.43,12.84,14.01,14.24,11.88



DOW max-min percentage point imbalance:
pct_sun    0.977
pct_mon    0.626
pct_tue    0.867
pct_wed    0.419
pct_thu    0.626
pct_fri    0.547
pct_sat    0.640
dtype: float64

Volume summary:


,n_ods,total_pnrs,avg_pnrs_per_od,median_pnrs_per_od,total_pax,avg_pax_per_od,median_pax_per_od
test_group,,,,,,,
Control,1022,1166146,1141.04,123.0,1166160,1141.06,123.0
T1_minus15,1039,1163922,1120.23,122.0,1163930,1120.24,122.0
T2_minus10,1018,1125168,1105.27,112.0,1125178,1105.28,112.0
T3_plus10,1021,1166242,1142.25,117.0,1166255,1142.27,117.0
T4_plus15,1015,1123291,1106.69,120.0,1123305,1106.70,120.0



PNR spread:
42,951 PNRs (3.74% of mean)

PAX spread:
42,950 PAX (3.74% of mean)


In [15]:
deployment_dates = pd.date_range(
    start=START_DATE,
    end=END_DATE,
    freq="D"
)

n_dates = len(deployment_dates)

pilot_plan = pilot_od_features.copy()

pilot_plan = (
    pilot_plan
    .groupby("test_group", group_keys=False)
    .apply(lambda x: x.sample(frac=1, random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

pilot_plan["date_idx"] = (
    pilot_plan
    .groupby("test_group")
    .cumcount()
    % n_dates
)

pilot_plan["Travel dates (Start)"] = pilot_plan["date_idx"].map(
    dict(enumerate(deployment_dates))
)

pilot_plan["Travel dates (End)"] = pilot_plan["Travel dates (Start)"]

pilot_plan["Loc1"] = (
    "P:" + pilot_plan["od_origin"].astype(str)
    + ",P:" + pilot_plan["od_destination"].astype(str)
)

pilot_plan["Comment"] = "Random Testing"
pilot_plan["Segment matches required"] = "FIRST"
pilot_plan["Result"] = pilot_plan["test_group"].map(TREATMENT_VALUE_MAP)

priority_plan = pilot_plan[
    [
        "Comment",
        "Segment matches required",
        "Travel dates (Start)",
        "Travel dates (End)",
        "Loc1",
        "Result"
    ]
].copy()

priority_plan["Travel dates (Start)"] = (
    pd.to_datetime(priority_plan["Travel dates (Start)"])
    .dt.strftime("%m/%d/%Y")
)

priority_plan["Travel dates (End)"] = (
    pd.to_datetime(priority_plan["Travel dates (End)"])
    .dt.strftime("%m/%d/%Y")
)

display(priority_plan)

C:\Users\939510\AppData\Local\Temp\1\ipykernel_10720\4275258343.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=1, random_state=RANDOM_SEED))


,Comment,Segment matches required,Travel dates (Start),Travel dates (End),Loc1,Result
0,Random Testing,FIRST,08/01/2026,08/01/2026,"P:TTN,P:AVL",1.00
1,Random Testing,FIRST,08/02/2026,08/02/2026,"P:ACT,P:MGM",1.00
2,Random Testing,FIRST,08/03/2026,08/03/2026,"P:SAV,P:GNV",1.00
3,Random Testing,FIRST,08/04/2026,08/04/2026,"P:DFW,P:HSV",1.00
4,Random Testing,FIRST,08/05/2026,08/05/2026,"P:GEG,P:SPI",1.00
...,...,...,...,...,...,...
5110,Random Testing,FIRST,08/03/2026,08/03/2026,"P:SMF,P:DSM",1.15
5111,Random Testing,FIRST,08/04/2026,08/04/2026,"P:EUG,P:MHK",1.15
5112,Random Testing,FIRST,08/05/2026,08/05/2026,"P:YQB,P:ICT",1.15
5113,Random Testing,FIRST,08/06/2026,08/06/2026,"P:GRI,P:FSM",1.15


In [16]:
priority_plan_spark = spark.createDataFrame(priority_plan)

priority_plan_spark.createOrReplaceTempView("priority_pilot_plan")

# priority_plan_spark.write.mode("overwrite").saveAsTable(
#     "rm_workspace.priority_pilot_plan"
# )

# priority_plan.to_csv(
#     "/dbfs/FileStore/priority_pilot_plan.csv",
#     index=False
# )